# Adversarial Patch: trained on two photos, evaluated on a third

Brown, Mané, Roy, Abadi, Gilmer, "Adversarial Patch" (2017), https://arxiv.org/abs/1712.09665, against a pretrained ResNet-50 (`IMAGENET1K_V1`).
Run: `pip install torch torchvision`, put `goldfish.jpg`, `dog.jpg`, `cat.jpg` next to this notebook, then execute the cells.

Unlike FGSM (one tiny step, invisible, per-image), the patch is trained once — with expectation-over-transformation (EOT) over its random position and scale, batched over goldfish + cat only — then pasted onto any photo, at any position, and still works. The dog photo is **held out**: the patch never sees it during training, so its flip is a genuine transfer test, not memorisation.

Outputs: `out/patch.png` (the trained patch, upscaled 3x), `out/goldfish_pXX.png` (goldfish with the patch at XX% of the image area), `out/dog_p10.png` (held-out dog with the same patch), `out/dog_clean.png` (dog without the patch, for comparison), `out/results.json` (top-3 labels, success rate over `eval-positions` random placements per sample, and the clean-dog baseline).

Photo sources: `goldfish.jpg` = Wikimedia Commons `Goldfish3.jpg` (public domain); `dog.jpg` = `Golden_Retriever_Dukedestiny01_drvd.jpg` (public domain); `cat.jpg` = `Cat_November_2010-1a.jpg` by Alvesgaspar (CC BY-SA 3.0 — attribute the author and share alike).

Target: `toaster` (Brown et al.'s classic example). An earlier, simpler version of this script (one training image per step, patch re-clamped on the forward pass instead of projected after the optimizer step) did not converge on `toaster` in 300/600 steps — goldfish stayed top-1 goldfish at 10%. That was an artifact of the optimization, not a property of the class: batching both training images per step and projecting the patch into `[0, 1]` after `opt.step()` (instead of clamping it on every forward pass) makes `toaster` converge cleanly, matching the paper's own example. See `README.md` for the numbers.

Exercises: 1) swap `target_name` for another class and compare how many steps it needs; 2) try a different `--size`-equivalent (`size_px`) and area range; 3) read about defenses: patch detection by saliency/gradient discontinuities, or Local Gradients Smoothing (Naseer et al., 2019).


In [ ]:
import json, random
from pathlib import Path
import torch, torch.nn.functional as F
from torchvision import models
from torchvision.io import read_image, write_png
from torchvision.transforms.functional import resize, center_crop

target_name, steps, size_px, model_name, eval_positions = "toaster", 500, 64, "resnet50", 20
torch.manual_seed(0)
random.seed(0)

weights = models.ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights).eval()
labels = weights.meta["categories"]
target = labels.index(target_name)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def logits(x):                                     # x: [B, 3, 224, 224] in [0, 1]
    return model((x - mean) / std)

def predict(x):
    return F.softmax(logits(x[None]), dim=1)[0]

def apply_patch(img, patch, area, pos=None):
    s = max(8, round((area ** 0.5) * img.shape[-1]))
    ptc = F.interpolate(patch[None], size=(s, s), mode="bilinear")[0]
    x0 = pos[0] if pos else random.randint(0, img.shape[-1] - s)
    y0 = pos[1] if pos else random.randint(0, img.shape[-2] - s)
    out = img.clone()
    out[:, y0:y0 + s, x0:x0 + s] = ptc
    return out

def success_rate(img, area, n):
    return sum(predict(apply_patch(img, patch, area)).argmax().item() == target for _ in range(n)) / n

photos = {name: center_crop(resize(read_image(f"{name}.jpg"), 256), 224).float() / 255
          for name in ["goldfish", "dog", "cat"]}

imgs = torch.stack([photos["goldfish"], photos["cat"]])
patch = torch.rand(3, size_px, size_px, requires_grad=True)
opt = torch.optim.Adam([patch], lr=0.05)
target_batch = torch.full((len(imgs),), target)
for step in range(steps):
    area = random.uniform(0.04, 0.22)
    batch = torch.stack([apply_patch(img, patch, area) for img in imgs])
    opt.zero_grad()
    loss = F.cross_entropy(logits(batch), target_batch)
    loss.backward()
    opt.step()
    with torch.no_grad():
        patch.clamp_(0, 1)
    if step % 50 == 0:
        print(step, float(loss))
patch = patch.detach()

out = Path("out"); out.mkdir(parents=True, exist_ok=True)
write_png((F.interpolate(patch[None], scale_factor=3, mode="nearest")[0] * 255).round().byte(), str(out / "patch.png"))

dog_clean = photos["dog"]
with torch.no_grad():
    clean_top = predict(dog_clean).topk(3)
write_png((dog_clean * 255).round().byte(), str(out / "dog_clean.png"))

results = []
for file, area in [("goldfish", 0.05), ("goldfish", 0.10), ("goldfish", 0.15), ("goldfish", 0.20), ("dog", 0.10)]:
    img = photos[file]
    s = max(8, round((area ** 0.5) * img.shape[-1]))
    pos = ((img.shape[-1] - s) // 2, (img.shape[-2] - s) // 2)
    adv = apply_patch(img, patch, area, pos)
    with torch.no_grad():
        top = predict(adv).topk(3)
    rate = success_rate(img, area, eval_positions)
    name = f"{file}_p{round(area * 100):02d}.png"
    write_png((adv * 255).round().byte(), str(out / name))
    results.append({
        "file": file, "area": area, "image": name,
        "top": [{"label": labels[i], "p": round(float(v), 4)} for v, i in zip(top.values, top.indices)],
        "success_rate": round(rate, 2),
    })
    print(file, name, results[-1]["top"][0], "success_rate", results[-1]["success_rate"])

(out / "results.json").write_text(json.dumps({
    "target": target_name, "steps": steps, "size_px": size_px, "model": model_name,
    "eval_positions": eval_positions, "train_images": ["goldfish", "cat"], "held_out": "dog",
    "dog_clean": {
        "image": "dog_clean.png",
        "top": [{"label": labels[i], "p": round(float(v), 4)} for v, i in zip(clean_top.values, clean_top.indices)],
    },
    "samples": results,
}, indent=2))
